<a href="https://colab.research.google.com/github/J3rmed/Estructura-de-Base-de-Datos/blob/main/Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Problema 1**

Dado un hash encontrar la secuencia de números que lo genera.
<br>Reglas:
* hash → sha 256.
* La secuencia de números son 10 enteros concatenados, cada uno en el rango [0,9].

In [13]:
import hashlib
import multiprocessing as mp
from tqdm.notebook import tqdm  # opcional, instalar con !pip install tqdm

def search_chunk(args):
    start, end, target_hash = args
    target = bytes.fromhex(target_hash)
    for i in range(start, end):
        s = f"{i:010d}"
        if hashlib.sha256(s.encode()).digest() == target:
            return s
    return None

def find_parallel(target_hash, num_processes=None):
    total = 10**10
    if num_processes is None:
        num_processes = mp.cpu_count()
    chunk_size = total // num_processes
    ranges = []
    for i in range(num_processes):
        start = i * chunk_size
        end = total if i == num_processes-1 else (i+1)*chunk_size
        ranges.append((start, end, target_hash))

    with mp.Pool(num_processes) as pool:
        # Usar imap_unordered para obtener resultados a medida que llegan
        for result in pool.imap_unordered(search_chunk, ranges):
            if result is not None:
                pool.terminate()
                return result
    return None

if __name__ == "__main__":
    target = input("Hash: ").strip()
    res = find_parallel(target)
    print("Secuencia:", res)

Hash: 07963a0301a24ee7906c98f1c965a7b6d58328a7f664ef89c92ce3001935ffac
Secuencia: 0001000000


# **Problema 2**

Dado el hash root del árbol de Merkle determinar el orden de las transacciones que lo generan.
<br>Reglas:

* Las transacciones son conocidas, y el número de transacciones es par.

In [8]:
import itertools
from functools import lru_cache

def hash_pair(left: bytes, right: bytes) -> bytes:
    """Calcula el hash de la concatenación de dos hashes (SHA256)."""
    return hashlib.sha256(left + right).digest()

def merkle_root(leaves):
    """
    Construye el árbol de Merkle para una lista de hojas (bytes) y devuelve la raíz (bytes).
    Asume que el número de hojas es potencia de 2 o al menos par.
    Si es impar, la última hoja se duplica (según la práctica común de Bitcoin, pero aquí se espera par).
    Para simplificar, solo trabajamos con listas de longitud par.
    """
    if len(leaves) == 1:
        return leaves[0]
    # Asegurar número par (para este problema ya es par)
    if len(leaves) % 2 != 0:
        leaves = leaves + [leaves[-1]]  # duplicar la última (caso general)
    next_level = []
    for i in range(0, len(leaves), 2):
        next_level.append(hash_pair(leaves[i], leaves[i+1]))
    return merkle_root(next_level)

def find_order(transactions_set, target_root_hex, max_permutations=None):
    """
    Busca la permutación de transactions_set que genera target_root_hex.
    transactions_set: lista de cadenas (transacciones) o bytes.
    target_root_hex: string hexadecimal del hash raíz.
    max_permutations: límite de permutaciones a probar (None = todas).
    Devuelve la lista ordenada (como strings) o None si no se encuentra.
    """
    # Convertir transacciones a bytes (hash SHA256 de la cadena, por ejemplo)
    # Asumimos que cada transacción ya viene como su hash (32 bytes) o como string.
    # Para simplificar, si es string, la convertimos a bytes usando .encode()
    leaves_bytes = []
    for tx in transactions_set:
        if isinstance(tx, str):
            # Podría ser el texto de la transacción; usualmente se usa su hash.
            # Aquí suponemos que la transacción ya es un hash hexadecimal.
            # Para un ejemplo claro, usaremos SHA256 del string.
            leaves_bytes.append(hashlib.sha256(tx.encode()).digest())
        else:
            leaves_bytes.append(tx)

    target_root = bytes.fromhex(target_root_hex)
    n = len(leaves_bytes)

    # Optimización con caché para la función merkle_root sobre tuplas (inmutable)
    @lru_cache(maxsize=None)
    def root_from_tuple(tup):
        return merkle_root(list(tup))

    # Probar permutaciones
    count = 0
    for perm in itertools.permutations(leaves_bytes):
        if root_from_tuple(perm) == target_root:
            # Devolver el orden original de las transacciones (como strings)
            # Mapeamos cada hash de vuelta a la transacción original
            # Para ello, construimos un diccionario (asumiendo hashes únicos)
            # Nota: Si hay transacciones repetidas, habrá ambigüedad.
            tx_map = {h: tx for h, tx in zip(leaves_bytes, transactions_set)}
            result = [tx_map[h] for h in perm]
            return result
        count += 1
        if max_permutations and count >= max_permutations:
            break
    return None

# Ejemplo de uso
if __name__ == "__main__":
    # Conjunto de transacciones (como strings)
    transacciones = ["A", "B", "C", "D"]
    # Construimos la raíz para un orden conocido, para probar
    # Primero convertimos a bytes (hash de cada string)
    leaves = [hashlib.sha256(tx.encode()).digest() for tx in transacciones]
    # Orden original: ["A","B","C","D"]
    root_conocida = merkle_root(leaves)
    print("Raíz para orden [A,B,C,D]:", root_conocida.hex())

    # Ahora, dado el conjunto desordenado y la raíz, buscamos el orden
    encontrado = find_order(transacciones, root_conocida.hex())
    print("Orden encontrado:", encontrado)

    # Prueba con otro orden
    otro_orden = ["B", "A", "D", "C"]
    leaves2 = [hashlib.sha256(tx.encode()).digest() for tx in otro_orden]
    root2 = merkle_root(leaves2)
    print("Raíz para orden [B,A,D,C]:", root2.hex())

    encontrado2 = find_order(transacciones, root2.hex())
    print("Orden encontrado para root2:", encontrado2)

Raíz para orden [A,B,C,D]: 1b3faa3fcc5ed50cd8592f805c6f8fce976b8582c739b26a6f3613b7f9b13617
Orden encontrado: ['A', 'B', 'C', 'D']
Raíz para orden [B,A,D,C]: cea2061b7cde1d598b770fb76d0fb5a300a3b032dec5d86c21cb1731832ed2a8
Orden encontrado para root2: ['B', 'A', 'D', 'C']
